# Proyecto Mundial: Exploracion inicial de datos medicos

Este notebook documenta la primera parte del proyecto: cargar el conjunto de datos, entender su estructura, revisar las features, calcular rangos de valores y visualizar frecuencias. La variable `condition` se trata como variable objetivo, por lo que los analisis de features la excluyen por defecto.


## 1. Preparacion del entorno

Ahora preparamos las librerias y las rutas del proyecto. Esto permite que el notebook funcione aunque se ejecute desde la carpeta del notebook o desde la raiz del repositorio.


In [ ]:
from pathlib import Path
import math

import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
elif (NOTEBOOK_DIR / "data" / "foot.csv").exists():
    PROJECT_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR / "mundial_project").exists():
    PROJECT_ROOT = NOTEBOOK_DIR / "mundial_project"
else:
    PROJECT_ROOT = NOTEBOOK_DIR.parent

DATA_PATH = PROJECT_ROOT / "data" / "foot.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "graficas_frecuencia"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH, OUTPUT_DIR


## 2. Carga de datos

Ahora estamos en la parte de analisis inicial: cargamos el CSV para comprobar dimensiones, columnas y primeras filas. Esto confirma que el dataset se lee correctamente antes de hacer cualquier transformacion o grafica.


In [ ]:
df = pd.read_csv(DATA_PATH)

print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
df.head()


## 3. Estructura del dataset

Ahora revisamos tipos de datos, valores no nulos y estadisticos generales. Esta exploracion sirve para detectar columnas numericas, posibles valores faltantes y escalas muy diferentes entre features.


In [ ]:
df.info()
df.describe(include="all")


## 4. Separacion de features y variable objetivo

Ahora definimos `condition` como la variable que queremos predecir mas adelante. Las demas columnas se consideran features, por eso las analizaremos sin mezclar la variable objetivo con las entradas del modelo.


In [ ]:
target = "condition"
features = [column for column in df.columns if column != target]

print(f"Variable objetivo: {target}")
print(f"Numero de features: {len(features)}")
features


## 5. Rangos de valores de las features

Ahora revisamos los rangos de cada feature para entender sus escalas, minimos, maximos y posibles anomalias. Esto es importante porque algunos modelos son sensibles a la escala y porque un rango inesperado puede indicar datos mal codificados.


In [ ]:
feature_ranges = []

for feature in features:
    series = df[feature]
    if pd.api.types.is_numeric_dtype(series):
        minimum = series.min()
        maximum = series.max()
        value_range = maximum - minimum
    else:
        minimum = None
        maximum = None
        value_range = None

    feature_ranges.append(
        {
            "feature": feature,
            "tipo_dato": str(series.dtype),
            "minimo": minimum,
            "maximo": maximum,
            "rango": value_range,
            "valores_unicos": series.nunique(dropna=True),
            "valores_nulos": series.isna().sum(),
        }
    )

rangos_features = pd.DataFrame(feature_ranges)
rangos_features


## 6. Funcion para graficar frecuencias

Ahora creamos una funcion reutilizable para visualizar cada feature. Si una columna numerica tiene muchos valores distintos usamos histograma; si tiene pocos valores distintos usamos barras, porque eso permite leer mejor variables discretas o categoricas codificadas.


In [ ]:
def plot_feature_frequency(series, feature_name, max_categories=20, bins=15, ax=None):
    current_ax = ax or plt.subplots(figsize=(8, 4.5))[1]
    clean_series = series.dropna()
    unique_count = clean_series.nunique()

    if pd.api.types.is_numeric_dtype(clean_series) and unique_count > max_categories:
        current_ax.hist(clean_series, bins=bins, edgecolor="black", color="#4C78A8")
        current_ax.set_xlabel(feature_name)
        current_ax.set_ylabel("Frecuencia")
    else:
        counts = clean_series.value_counts().sort_index()
        counts.plot(kind="bar", ax=current_ax, color="#59A14F", edgecolor="black")
        current_ax.set_xlabel(feature_name)
        current_ax.set_ylabel("Frecuencia")
        current_ax.tick_params(axis="x", rotation=45)

    current_ax.set_title(f"Frecuencia de {feature_name}")
    current_ax.grid(axis="y", alpha=0.25)
    return current_ax


## 7. Graficas individuales

Ahora calculamos las frecuencias feature por feature. Estas graficas ayudan a detectar distribuciones desequilibradas, valores raros, variables binarias y posibles columnas que necesiten transformacion antes de entrenar modelos.


In [ ]:
for feature in features:
    plot_feature_frequency(df[feature], feature)
    plt.tight_layout()
    plt.show()


## 8. Grafica combinada

Ahora juntamos todas las frecuencias en una sola figura. Esta vista sirve para comparar rapidamente las distribuciones y decidir que features necesitan una revision mas detallada.


In [ ]:
cols = 3
rows = math.ceil(len(features) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 5.2, rows * 3.8))
flat_axes = axes.ravel() if hasattr(axes, "ravel") else [axes]

for ax, feature in zip(flat_axes, features):
    plot_feature_frequency(df[feature], feature, ax=ax)

for ax in flat_axes[len(features):]:
    ax.set_visible(False)

fig.suptitle("Graficas de frecuencia por feature", fontsize=16)
fig.tight_layout(rect=(0, 0, 1, 0.98))
plt.show()


## 9. Guardado de resultados

Ahora guardamos las graficas en `outputs/graficas_frecuencia` para que el analisis sea reproducible y pueda entregarse junto al proyecto sin depender solo de la visualizacion del notebook.


In [ ]:
def clean_filename(name):
    valid = [char if char.isalnum() or char in ("-", "_") else "_" for char in name]
    return "".join(valid).strip("_") or "feature"

for feature in features:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    plot_feature_frequency(df[feature], feature, ax=ax)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"{clean_filename(feature)}_frecuencia.png", dpi=160)
    plt.close(fig)

fig, axes = plt.subplots(rows, cols, figsize=(cols * 5.2, rows * 3.8))
flat_axes = axes.ravel() if hasattr(axes, "ravel") else [axes]

for ax, feature in zip(flat_axes, features):
    plot_feature_frequency(df[feature], feature, ax=ax)

for ax in flat_axes[len(features):]:
    ax.set_visible(False)

fig.suptitle("Graficas de frecuencia por feature", fontsize=16)
fig.tight_layout(rect=(0, 0, 1, 0.98))
fig.savefig(OUTPUT_DIR / "todas_las_frecuencias.png", dpi=180)
plt.close(fig)

print(f"Graficas guardadas en: {OUTPUT_DIR.resolve()}")
